# Fase 5 — Agente ReAct (Colab)

Notebook para correr el agente ReAct sobre HotpotQA en GPU de Google Colab.

## Antes de empezar

1. **Subir el proyecto a Google Drive** — carpeta con la siguiente estructura mínima:
   ```
   tp-multihop-qa/
   ├── src/
   ├── requirements.txt
   ├── data/
   │   ├── validation_subset_500.json
   │   └── faiss_index/
   │       ├── index.faiss
   │       └── index.pkl
   └── results/
       └── predictions_agent.json   ← opcional, para retomar donde se dejó
   ```
2. **Activar GPU**: Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)
3. **Token de Hugging Face**: necesitás haber aceptado la licencia de `google/gemma-4-E2B-it`.
   Guardalo en Colab Secrets como `HF_TOKEN` (ícono de llave en el panel izquierdo).

## Estrategia de guardado

- Los resultados se guardan en `/content/` después de **cada pregunta** (rápido).
- Cada `DRIVE_SYNC_EVERY` preguntas se sincronizan a Drive (resiliente ante cortes de sesión).
- Si la sesión se corta, al volver a correr el notebook retoma automáticamente desde Drive.


In [ ]:
#@title 1. Verificar GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '⚠️  No se detectó GPU. Activá GPU en Entorno de ejecución.')

import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    vram = gpu.total_memory / 1e9
    print(f'\n✓ GPU: {gpu.name}')
    print(f'✓ VRAM: {vram:.1f} GB')
    if vram < 10:
        print('⚠️  VRAM puede ser insuficiente para Gemma 4 E2B en bfloat16 (~10 GB).')
else:
    print('\n⚠️  CUDA no disponible. Verificá la configuración de GPU.')


In [ ]:
#@title 2. Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('✓ Drive montado en /content/drive')


In [ ]:
#@title 3. Configuración { display-mode: "form" }

# Ruta a la carpeta del proyecto en tu Google Drive
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/tp-multihop-qa"  #@param {type:"string"}

# Top-k documentos por búsqueda del agente
K_RETRIEVAL = 5  #@param {type:"integer"}

# Sincronizar resultados a Drive cada N preguntas (0 = solo al final)
DRIVE_SYNC_EVERY = 10  #@param {type:"integer"}

# Limitar a N preguntas para prueba rápida (0 = todas)
LIMIT = 0  #@param {type:"integer"}

import os
from pathlib import Path

drive_path = Path(DRIVE_PROJECT_PATH)
if not drive_path.exists():
    raise FileNotFoundError(f'No se encontró el proyecto en: {DRIVE_PROJECT_PATH}\nVerificá la ruta en la configuración.')

print(f'✓ Proyecto encontrado en: {DRIVE_PROJECT_PATH}')

for required in ['src', 'data/validation_subset_500.json', 'data/faiss_index', 'requirements.txt']:
    p = drive_path / required
    status = '✓' if p.exists() else '✗  FALTA'
    print(f'  {status}: {required}')

partial = drive_path / 'results' / 'predictions_agent.json'
if partial.exists():
    import json
    with open(partial) as f:
        n_done = len(json.load(f))
    print(f'\n✓ Predicciones parciales encontradas en Drive: {n_done} preguntas ya procesadas.')
    print('  El notebook retomará desde donde se dejó.')
else:
    print('\n  (No hay predicciones previas — comenzará desde cero.)')


In [ ]:
#@title 4. Copiar proyecto a /content/
import shutil
from pathlib import Path

CONTENT_DIR = Path('/content/tp-multihop-qa')
drive_path = Path(DRIVE_PROJECT_PATH)

# Copiar src/ y requirements.txt (código fuente)
for item in ['src', 'requirements.txt']:
    dest = CONTENT_DIR / item
    if dest.exists():
        if dest.is_dir():
            shutil.rmtree(dest)
        else:
            dest.unlink()
    src_item = drive_path / item
    if src_item.is_dir():
        shutil.copytree(src_item, dest)
    else:
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src_item, dest)
    print(f'✓ Copiado: {item}')

# Copiar datos
data_dest = CONTENT_DIR / 'data'
data_dest.mkdir(parents=True, exist_ok=True)

subset_src = drive_path / 'data' / 'validation_subset_500.json'
subset_dest = data_dest / 'validation_subset_500.json'
shutil.copy(subset_src, subset_dest)
print('✓ Copiado: data/validation_subset_500.json')

faiss_src = drive_path / 'data' / 'faiss_index'
faiss_dest = data_dest / 'faiss_index'
if faiss_dest.exists():
    shutil.rmtree(faiss_dest)
shutil.copytree(faiss_src, faiss_dest)
print('✓ Copiado: data/faiss_index/')

# Copiar predicciones parciales si existen (resume)
results_dest = CONTENT_DIR / 'results'
results_dest.mkdir(parents=True, exist_ok=True)
partial_src = drive_path / 'results' / 'predictions_agent.json'
partial_dest = results_dest / 'predictions_agent.json'
if partial_src.exists():
    shutil.copy(partial_src, partial_dest)
    print('✓ Copiado: results/predictions_agent.json (resume)')

print(f'\n✓ Proyecto listo en {CONTENT_DIR}')


In [ ]:
#@title 5. Instalar dependencias
import subprocess, sys
from pathlib import Path

req_path = Path('/content/tp-multihop-qa/requirements.txt')
print('Instalando dependencias (puede tardar 2-3 minutos)...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req_path)],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])
    raise RuntimeError('Error instalando dependencias.')
print('✓ Dependencias instaladas.')


In [ ]:
#@title 6. Login a Hugging Face
from google.colab import userdata
from huggingface_hub import login

try:
    token = userdata.get('HF_TOKEN')
    login(token=token, add_to_git_credential=False)
    print('✓ Login con token desde Colab Secrets.')
except Exception:
    print('Token no encontrado en Secrets. Ingresá el token manualmente:')
    login()


In [ ]:
#@title 7. Ejecutar Fase 5 — Agente ReAct
import json
import shutil
import sys
from pathlib import Path

from tqdm.notebook import tqdm

CONTENT_DIR = Path('/content/tp-multihop-qa')
sys.path.insert(0, str(CONTENT_DIR))

from src import config

# Override de paths para que apunten a /content/
config.SUBSET_PATH     = CONTENT_DIR / 'data' / 'validation_subset_500.json'
config.VECTOR_STORE_DIR = CONTENT_DIR / 'data' / 'faiss_index'
config.RESULTS_DIR     = CONTENT_DIR / 'results'
config.ensure_dirs()

from src.agent import run_agent
from src.data_utils import load_subset
from src.evaluation import evaluate
from src.llm import load_model
from src.vector_store import get_embeddings, load_vector_store

RESULTS_LOCAL = CONTENT_DIR / 'results' / 'predictions_agent.json'
RESULTS_DRIVE = Path(DRIVE_PROJECT_PATH) / 'results' / 'predictions_agent.json'


def load_existing(path):
    if path.exists():
        with open(path) as f:
            return {r['id']: r for r in json.load(f)}
    return {}


def save_local(records):
    RESULTS_LOCAL.parent.mkdir(parents=True, exist_ok=True)
    with open(RESULTS_LOCAL, 'w') as f:
        json.dump(list(records.values()), f, ensure_ascii=False, indent=2)


def sync_to_drive(records):
    save_local(records)
    RESULTS_DRIVE.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(RESULTS_LOCAL, RESULTS_DRIVE)


# ── Cargar datos ─────────────────────────────────────────────────────────────
subset = load_subset(config.SUBSET_PATH)
if LIMIT > 0:
    subset = subset[:LIMIT]

done = load_existing(RESULTS_LOCAL)
pending = [ex for ex in subset if ex['id'] not in done]
print(f'Total: {len(subset)} | Ya procesadas: {len(done)} | Pendientes: {len(pending)}')

if not pending:
    print('\n✓ Todas las preguntas ya están procesadas. Pasá a la celda de Resultados.')
else:
    # ── Cargar modelos ────────────────────────────────────────────────────────
    print('\nCargando índice FAISS...')
    embeddings = get_embeddings()
    vs = load_vector_store(config.VECTOR_STORE_DIR, embeddings=embeddings)
    print('✓ Índice cargado.')

    print(f'\nCargando modelo: {config.LLM_MODEL}')
    model, tokenizer = load_model()

    # ── Loop de inferencia ────────────────────────────────────────────────────
    print(f'\nIniciando inferencia (max_steps={config.AGENT_MAX_STEPS}, k={K_RETRIEVAL})...\n')

    for i, ex in enumerate(tqdm(pending, desc='Agente ReAct')):
        result = run_agent(
            model, tokenizer, vs,
            question=ex['question'],
            max_steps=config.AGENT_MAX_STEPS,
            k=K_RETRIEVAL,
        )
        done[ex['id']] = {
            'id':       ex['id'],
            'question': ex['question'],
            'gold':     ex['answer'],
            'pred':     result['answer'],
            'type':     ex.get('type', 'unknown'),
            'level':    ex.get('level', 'unknown'),
            'steps':    result['steps'],
            'finished': result['finished'],
            'trace':    result['trace'],
        }

        # Guardar localmente tras cada pregunta
        save_local(done)

        # Sincronizar a Drive cada DRIVE_SYNC_EVERY preguntas
        if DRIVE_SYNC_EVERY > 0 and (i + 1) % DRIVE_SYNC_EVERY == 0:
            sync_to_drive(done)
            tqdm.write(f'  ✓ Sync a Drive: {len(done)} predicciones guardadas.')

    # Sync final
    sync_to_drive(done)
    print(f'\n✓ Completado. {len(done)} predicciones guardadas en Drive.')


In [ ]:
#@title 8. Resultados
import json
import sys
from pathlib import Path

CONTENT_DIR = Path('/content/tp-multihop-qa')
sys.path.insert(0, str(CONTENT_DIR))

from src.evaluation import evaluate

results_path = CONTENT_DIR / 'results' / 'predictions_agent.json'
with open(results_path) as f:
    records = json.load(f)

m = evaluate(records)

finished_count = sum(1 for r in records if r.get('finished'))
avg_steps = sum(r.get('steps', 0) for r in records) / len(records)

search_calls = sum(
    sum(1 for s in r.get('trace', []) if s.get('action_type') == 'search')
    for r in records
)
lookup_calls = sum(
    sum(1 for s in r.get('trace', []) if s.get('action_type') == 'lookup')
    for r in records
)
questions_with_lookup = sum(
    1 for r in records
    if any(s.get('action_type') == 'lookup' for s in r.get('trace', []))
)

print('=' * 55)
print('Resultados — Agente ReAct')
print('=' * 55)
print(f"  N    : {m['n']}")
print(f"  EM   : {m['em']*100:.1f}%")
print(f"  F1   : {m['f1']*100:.1f}%")
print()
print(f"  Episodios con finish[]: {finished_count}/{m['n']} ({finished_count/m['n']*100:.1f}%)")
print(f"  Pasos promedio        : {avg_steps:.1f}")
print()
print(f"  Llamadas search[]     : {search_calls}")
print(f"  Llamadas lookup[]     : {lookup_calls}")
print(f"  Preguntas con lookup[]: {questions_with_lookup}/{m['n']} ({questions_with_lookup/m['n']*100:.1f}%)")
print()
print('Por type:')
for t, v in sorted(m['by_type'].items()):
    print(f"  {t:>12}: EM={v['em']*100:.1f}%  F1={v['f1']*100:.1f}%")


In [ ]:
#@title 9. Backup final a Drive
import shutil
from pathlib import Path

CONTENT_DIR = Path('/content/tp-multihop-qa')
drive_results = Path(DRIVE_PROJECT_PATH) / 'results'
drive_results.mkdir(parents=True, exist_ok=True)

src_file = CONTENT_DIR / 'results' / 'predictions_agent.json'
dst_file = drive_results / 'predictions_agent.json'

if src_file.exists():
    shutil.copy(src_file, dst_file)
    size_mb = dst_file.stat().st_size / 1e6
    print(f'✓ Backup completo: {dst_file}')
    print(f'  Tamaño: {size_mb:.1f} MB')
else:
    print('⚠️  No se encontró el archivo de predicciones local.')
